In [ ]:
!pip install transformers datasets torch scikit-learn pandas tqdm -q

from google.colab import drive
drive.mount('/content/drive')

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU not available. This notebook is intended for a Colab T4 runtime.')


In [ ]:
import io
import re

import pandas as pd
from google.colab import files
from IPython.display import display
from sklearn.model_selection import train_test_split

TEXT_KEYWORDS = ('text', 'review', 'sentence', 'content', 'tweet')
LABEL_KEYWORDS = ('label', 'sentiment', 'rating', 'score', 'polarity')
LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
MAX_SAMPLE_SIZE = 200_000


def _normalize_column_name(column_name: str) -> str:
    return re.sub(r'[^a-z0-9]+', ' ', str(column_name).strip().lower()).strip()


def _keyword_score(column_name: str, keywords: tuple[str, ...]) -> int:
    normalized_name = _normalize_column_name(column_name)
    tokens = set(normalized_name.split())
    score = 0
    for keyword in keywords:
        if normalized_name == keyword:
            score += 10
        if keyword in tokens:
            score += 6
        elif keyword in normalized_name:
            score += 2
    return score


def detect_columns(df: pd.DataFrame) -> tuple[str, str]:
    text_candidates = []
    label_candidates = []

    for column_name in df.columns:
        series = df[column_name]
        text_score = _keyword_score(column_name, TEXT_KEYWORDS)
        if text_score > 0:
            if pd.api.types.is_string_dtype(series) or pd.api.types.is_object_dtype(series):
                text_score += 3
            avg_length = float(series.dropna().astype(str).str.len().head(500).mean() or 0.0)
            text_candidates.append((text_score, avg_length, column_name))

        label_score = _keyword_score(column_name, LABEL_KEYWORDS)
        if label_score > 0:
            unique_values = float(series.dropna().nunique())
            if pd.api.types.is_numeric_dtype(series):
                label_score += 3
            if unique_values <= 10:
                label_score += 2
            label_candidates.append((label_score, -unique_values, column_name))

    if not text_candidates:
        raise ValueError(f'Could not auto-detect the text column. Available columns: {list(df.columns)}')
    if not label_candidates:
        raise ValueError(f'Could not auto-detect the label column. Available columns: {list(df.columns)}')

    text_candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
    label_candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)

    text_col = text_candidates[0][2]
    label_col = label_candidates[0][2]

    if text_col == label_col:
        alternatives = [candidate[2] for candidate in label_candidates if candidate[2] != text_col]
        if not alternatives:
            raise ValueError('Detected the same column for text and label. Please inspect the CSV manually.')
        label_col = alternatives[0]

    return text_col, label_col


def normalize_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        normalized = value.strip().lower()
        mapping = {
            'negative': 0,
            'neg': 0,
            'neutral': 1,
            'neu': 1,
            'positive': 2,
            'pos': 2,
        }
        if normalized in mapping:
            return mapping[normalized]
        try:
            return int(float(normalized))
        except ValueError:
            return None
    try:
        return int(value)
    except Exception:
        return None


uploaded = files.upload()
if not uploaded:
    raise ValueError('Please upload exactly one CSV file with text and label columns.')

uploaded_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[uploaded_name]), low_memory=False)

text_col, label_col = detect_columns(df)
print(f'Detected text column: {text_col}')
print(f'Detected label column: {label_col}')

working_df = df[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).copy()
working_df['text'] = working_df['text'].fillna('').astype(str).str.strip()
working_df['label'] = working_df['label'].map(normalize_label)
working_df = working_df.dropna(subset=['text', 'label'])
working_df = working_df[working_df['text'].str.len() > 0]
working_df['label'] = working_df['label'].astype(int)
working_df = working_df[working_df['label'].isin([0, 1, 2])].reset_index(drop=True)

print('Full dataset shape:', working_df.shape)
print('\nLabel distribution (full dataset):')
print(working_df['label'].value_counts().sort_index())
print('\nSample rows:')
display(working_df.head())

if len(working_df) > MAX_SAMPLE_SIZE:
    sampled_df, _ = train_test_split(
        working_df,
        train_size=MAX_SAMPLE_SIZE,
        stratify=working_df['label'],
        random_state=SEED,
    )
    sampled_df = sampled_df.reset_index(drop=True)
else:
    sampled_df = working_df.copy()

print(f'\nUsing {len(sampled_df):,} rows for fine-tuning.')
print('Label distribution (sampled dataset):')
print(sampled_df['label'].value_counts(normalize=True).sort_index().round(4))


In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

train_df, val_df = train_test_split(
    sampled_df,
    test_size=0.1,
    stratify=sampled_df['label'],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)


class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.labels = list(labels)
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=max_length,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx], dtype=torch.long) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = SentimentDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer)
val_dataset = SentimentDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer)

print('Train split shape:', train_df.shape)
print('Validation split shape:', val_df.shape)
print('Tokenized train samples:', len(train_dataset))
print('Tokenized validation samples:', len(val_dataset))
print('Sample item keys:', train_dataset[0].keys())


In [ ]:
import inspect
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer

model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro'),
    }


training_kwargs = {
    'output_dir': '/content/results',
    'num_train_epochs': 3,
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 32,
    'warmup_steps': 500,
    'weight_decay': 0.01,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'fp16': torch.cuda.is_available(),
    'logging_steps': 100,
    'save_total_limit': 2,
    'report_to': 'none',
    'metric_for_best_model': 'f1_macro',
    'greater_is_better': True,
    'seed': SEED,
}

strategy_parameter = 'eval_strategy' if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters else 'evaluation_strategy'
training_kwargs[strategy_parameter] = 'epoch'

args = TrainingArguments(**training_kwargs)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

print(args)


In [ ]:
import time

print('Starting training...')
start_time = time.time()
train_output = trainer.train()
training_time_seconds = time.time() - start_time
evaluation_results = trainer.evaluate()

print(f"Final accuracy: {evaluation_results.get('eval_accuracy', 0.0):.4f}")
print(f"Final F1: {evaluation_results.get('eval_f1_macro', 0.0):.4f}")
print(f'Training time: {training_time_seconds / 60:.2f} minutes')


In [ ]:
import json
from pathlib import Path

save_dir = Path('/content/drive/MyDrive/ReviewSense_RoBERTa')
save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

with open(save_dir / 'label_map.json', 'w', encoding='utf-8') as label_file:
    json.dump(LABEL_MAP, label_file, indent=2)

print('Model saved to Google Drive! Download ReviewSense_RoBERTa folder.')
print(f'Saved path: {save_dir}')


In [ ]:
import numpy as np
import torch

test_reviews = [
    ('Positive', 'This phone is amazing, fast, and worth every rupee.'),
    ('Negative', 'Worst purchase ever. It stopped working after two days.'),
    ('Neutral', 'The product arrived on time and works as described.'),
    ('Hindi translated', 'Hindi translated: This product is very good and easy to use.'),
    ('Sarcastic', 'Fantastic, another charger that dies in a week. Exactly what I needed.'),
]

model.eval()
device = next(model.parameters()).device

for review_type, review_text in test_reviews:
    encoded = tokenizer(
        review_text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128,
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded).logits

    probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    predicted_label = int(np.argmax(probabilities))
    confidence = float(probabilities[predicted_label])

    print(f'[{review_type}] {review_text}')
    print(f"Predicted label: {LABEL_MAP[predicted_label]} ({predicted_label}) | Confidence: {confidence:.4f}")
    print('-' * 100)
